## Pré-processamento

Aqui os dados minuto a minuto viram as duas tabelas usadas nas próximas etapas:

- **`dias`**: uma linha por pessoa-dia, com a glicose do dia, o comportamento (carboidratos, proteína, gordura, fibra, atividade e horário da última refeição) e os dados do `bio.csv`;
- **`dataset_rna`**: os exemplos de treino da rede, em que a entrada é a glicose dos dias anteriores + o comportamento do dia + o bio, e o alvo é a glicose média do dia.

As decisões da EDA valem aqui: sensor Dexcom, dia válido com cobertura ≥ 90% e exclusão dos participantes 007, 018 e 026.

In [11]:
# configurações
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
# importa as funções do src/dados.py
from src.dados import carregar, listar_pids, resumo_dia, maior_sequencia

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Agora testando a validade das funções para confirmar as informações presentes nos dados que iremos processar

In [12]:
# teste das funções
pids = listar_pids()
print(len(pids), pids[:5])

df = carregar(1)
print(df.shape)
df.columns.tolist()

45 [1, 2, 3, 4, 5]
(14730, 16)


['Unnamed: 0',
 'Timestamp',
 'Libre GL',
 'Dexcom GL',
 'HR',
 'Calories (Activity)',
 'METs',
 'Meal Type',
 'Calories',
 'Carbs',
 'Protein',
 'Fat',
 'Fiber',
 'Amount Consumed',
 'Image path',
 'pid']

Os números e os nomes das colunas estão OK.

Agora para o resumo de cada dia

Os arquivos têm uma linha por minuto, mas a rede prevê a glicose de um dia e o AG recomenda o comportamento de cada dia. A função `resumo_dia` transforma os minutos de um participante em uma linha por dia, com a cobertura do Dexcom (para filtrar os dias válidos), a glicose (média, desvio, pico e % acima de 140 mg/dL) e as seis variáveis de comportamento: carboidratos, proteína, gordura, fibra, atividade e horário da última refeição. Usar dias também evita o vazamento dos valores interpolados minuto a minuto. Na atividade, `min_count=1` faz um dia sem registro do Fitbit virar vazio, e não 0.

In [13]:
# a função resumo_dia está em src/dados.py; aqui testamos no participante 001
resumo_dia(df)

,data,cobertura,glic_media,glic_dp,glic_pico,pct_140,carbs,proteina,gordura,fibra,n_refeicoes,atividade,hora_ultima_refeicao,pid
0,2020-05-01,0.514583,119.259109,13.432144,176.0,6.072874,103.0,88.0,54.2,12.0,2,1960.298337,20.800000,1
1,2020-05-02,0.996528,97.637631,17.872912,142.0,0.418118,220.0,43.0,76.5,7.0,5,3297.322870,19.866667,1
2,2020-05-03,0.996528,96.742160,11.871717,132.0,0.000000,182.0,152.0,57.5,6.0,6,3149.183949,22.433333,1
3,2020-05-04,1.000000,104.495833,12.702486,144.0,0.972222,210.0,47.0,74.0,3.0,5,2565.539679,21.783333,1
4,2020-05-05,1.000000,101.417361,9.145802,126.0,0.000000,116.0,120.0,67.0,11.0,4,3060.174796,21.300000,1
5,2020-05-06,1.000000,103.157639,9.100765,129.0,0.000000,191.0,118.0,63.5,6.0,4,2873.349914,20.966667,1
6,2020-05-07,1.000000,109.981250,10.588680,160.0,1.736111,160.0,62.9,39.5,12.0,4,3238.088277,21.833333,1
7,2020-05-08,1.000000,110.192361,10.551206,138.0,0.000000,178.0,161.7,56.5,12.0,5,3483.099349,22.483333,1
8,2020-05-09,1.000000,113.291667,13.405409,155.0,4.861111,137.0,63.0,67.0,12.0,4,3307.387516,18.683333,1
9,2020-05-10,1.000000,112.961806,9.685295,143.0,1.597222,144.0,66.0,30.5,0.0,3,2545.095884,19.983333,1


No participante 001, os 14.730 minutos viram 11 linhas, uma por dia. Os dias da ponta aparecem com cobertura baixa (0,51 em 01/05 e 0,40 em 11/05) e também com somas menores, como no último dia, que tem uma única refeição e só 73 g de carboidrato. Eles ficam de fora pela regra dos 90%. Os 9 dias inteiros têm cobertura perto de 1 e glicose média entre 97 e 113 mg/dL, o esperado para alguém do grupo saudável. Como o dia vai da meia-noite às 23h59, um lanche depois da meia-noite conta para o dia seguinte. Por isso, a `hora_ultima_refeicao` pode subestimar o quanto a pessoa comeu tarde, um limite que aceitamos para manter o dia igual ao da glicose.

### Todos os participantes

A `resumo_dia` foi para o `src/dados.py`, junto com a `maior_sequencia`. Na mudança, ela passou a completar o calendário: uma data sem nenhuma linha no arquivo entra com cobertura 0, para que a contagem de dias seguidos não pule o buraco. Agora ela roda para os 45 participantes, e as regras da EDA (dia bom com cobertura ≥ 90% e pelo menos 9 dias bons seguidos) separam os válidos.

In [14]:
# resumo de todos os 45 participantes
todos = pd.concat([resumo_dia(carregar(p)) for p in listar_pids()], ignore_index=True)

# dia bom = Dexcom com valor em pelo menos 90% dos minutos
todos['bom'] = todos['cobertura'] >= 0.9

# participante válido = pelo menos 9 dias bons seguidos
seq = todos.groupby('pid')['bom'].apply(maior_sequencia)
validos = seq[seq >= 9].index.tolist()
excluidos = seq[seq < 9].index.tolist()
print('excluídos:', excluidos, '| válidos:', len(validos))

excluídos: [7, 18, 26] | válidos: 42


In [15]:
# tabela final de dias, só com os participantes válidos
dias = todos[todos['pid'].isin(validos)].reset_index(drop=True)
print(len(dias), 'dias no calendário,', dias['bom'].sum(), 'dias bons')

# dias bons com algum valor vazio
dias[dias['bom']].isna().sum()[lambda s: s > 0]

493 dias no calendário, 378 dias bons


atividade    1
dtype: int64

O resumo dos 45 participantes chega ao mesmo resultado da EDA: o 007, o 018 e o 026 não têm 9 dias bons seguidos e ficam de fora. A tabela `dias` fica com os 42 válidos, somando 493 dias no calendário, dos quais 378 são bons. Entre esses dias bons, só um tem valor vazio: a atividade do dia em que o participante 033 não usou o Fitbit. Na EDA ele aparecia como 0 caloria, mas agora, com o `min_count=1`, aparece como vazio, o que é mais honesto, porque não sabemos quanto a pessoa se mexeu. Esse dia, junto com os dias com carboidrato muito alto, é tratado na próxima etapa.